# Web APIs: concepts and first setup
### Python · FastAPI · Windows PowerShell

**Part A** explains how a client communicates with a server. **Part B** creates a small local API and saves its source code on GitHub.

**How to use this notebook**

- Read the notebook in a notebook viewer, GitHub, or VS Code with Microsoft's Jupyter extension.
- All cells are explanatory Markdown. Commands belong in **PowerShell**; Python belongs in the named **`.py` file**. A notebook kernel is not required.
- Run commands one line at a time. Do not copy output examples into the terminal.
- Follow Part B in order. Each numbered section includes a result to check.
- Commands target Windows. macOS and Linux use different executable paths and installation steps.

**Contents**

1. Part A: browser, client, server, HTTP, APIs, routes, JSON, validation, and development tools.
2. Demo: observe the completed application.
3. Part B: software checks, GitHub, folders, environments, packages, an API, requests, and Git.
4. Troubleshooting, resuming work, and reference links.

# Part A · Concepts
## A1. A page can ask another program for help

Imagine a page with a name box and a **Greet** button. Entering `Alex` and clicking the button displays `Hello, Alex!`.

In this example, the page sends the name to a Python program. Python builds the greeting and sends it back. The page then displays the returned message. A click has caused two programs to communicate.

The screen is only the visible part of this interaction. It accepts input and displays the result; the Python program performs the requested work.

The names for these parts are:

- A **browser**, such as Chrome, Firefox, or Edge, loads pages, runs JavaScript, and can send requests.
- The **frontend** is the part of an application presented to the user. HTML describes its content, CSS controls its appearance, and JavaScript handles interactions such as the button click.
- A **client** is a program acting as the requester. In this exchange, the browser is the client.

The person clicks the button; the browser sends the request. A terminal command or another Python program can also act as a client. Neither needs a visible button to ask for the same greeting.

## A2. Written instructions need a running program

A recipe can remain in a drawer all day without producing a meal. Someone has to follow its instructions before anything happens.

A saved Python file has the same distinction: its instructions exist on disk, but they only do work when executed. For our greeting example, a program starts, waits for incoming requests, and handles each one as it arrives. After returning a greeting, it remains available for the next request.

A **process** is a running instance of a program. A **server** receives requests and sends responses. Here, “start the server” means start the software process that makes the application available. The word server can also refer to the computer hosting that software.

The **backend** handles requests, applies rules, and can read or store data. Our Python greeting code is part of the backend. It runs separately from the browser.

### Both programs can be on the same computer

During development, one computer can run the browser and the server at the same time. The browser still sends a request to the separate server process.

The address **`127.0.0.1`** points back to the computer making the request. **`localhost`** normally resolves to a loopback address too. Opening `localhost` on a second computer points to that second computer.

Closing a browser tab leaves the server process running. Stopping the server leaves the Python files on disk, but the browser can no longer obtain new responses from that server.

## A3. Two programs need an agreed way to communicate

Suppose a service desk accepts written requests. Each request identifies the destination, describes the action, and includes any information needed to carry it out. The reply says what happened and includes the result when there is one.

This arrangement works because both sides understand the format. A computer needs that agreement too: a sequence of received bytes must be recognisable as an instruction and its inputs.

**HTTP**, Hypertext Transfer Protocol, supplies a shared set of rules for requests and responses. The client sends an HTTP request. The server reads it, handles it, and sends an HTTP response.

| Everyday purpose | HTTP component |
|---|---|
| State the action | Request method |
| Identify the destination and operation | Target URL |
| Describe the message, such as its format | Headers |
| Include submitted information | Optional request body |
| Report the outcome | Response status code |
| Return the result | Optional response body |

A **header** is a named piece of information about the message. For example, `Content-Type: application/json` tells the receiver how to interpret the body's data. The body is the message's content; we will examine JSON shortly.

For the greeting, the submitted information is `Alex`, and the returned result contains `Hello, Alex!`. Other responses might contain a web page, an image, or a file.

**HTTPS** protects HTTP communication with encryption and server authentication through TLS. The local lab uses HTTP.

For example, suppose we want the server to produce a greeting for Alex. The client sends a request to `http://localhost:8000/greet`:

```http
POST /greet HTTP/1.1
Host: localhost:8000
Content-Type: application/json

{"name":"Alex"}
```

Read it line by line:

- `POST` says we are submitting information for the server to process. This is the **request method**.
- `/greet` identifies the operation we want to use. Together with the server address, it forms the **target URL**: `http://localhost:8000/greet`.
- `HTTP/1.1` identifies the version of HTTP used for this message.
- `Host: localhost:8000` specifies the server’s address.
- `Content-Type: application/json` says that the information below is written in a format called JSON, which we will examine shortly. This line and the `Host` line are **headers**: named details that help the server interpret the request.
- After the blank line, `{"name":"Alex"}` supplies the actual information: the name is Alex. This is the **request body**.

The server’s reply can also contain headers and a body. For this example, that part of the reply could look like this:

```http
Content-Type: application/json

{"message":"Hello, Alex!"}
```

`Content-Type` tells the client that the returned information uses JSON. After the blank line comes `{"message":"Hello, Alex!"}`, which contains the greeting. This is the **response body**.

These examples show selected parts of the messages. We will cover response status codes separately.

## A4. An address needs more than a building number

Imagine delivering a request to a building that contains several service desks. Reaching the building is the first step. The request must also reach the intended desk and identify the service wanted there.

A computer can run several network services at once. Its address identifies the computer; a numbered port helps direct communication to the service listening there. Once that service receives a request, the application reads the requested path.

These pieces appear together in a **URL**, an address used by the client:

```text
http://127.0.0.1:8766/hello?name=Alex
```

| Part | Meaning in this request |
|---|---|
| `http` | Use HTTP to communicate |
| `127.0.0.1` | Contact this computer |
| `8766` | Contact the service listening on this port |
| `/hello` | Request the application's greeting operation |
| `?name=Alex` | Supply the additional input `name=Alex` |

A **port** is a numbered network endpoint. The client must use the port on which the server is listening. `8766` is a choice for this lab; it has no special meaning to FastAPI.

The **path** identifies what is requested within the application. It does not have to correspond to a file on disk. The part after `?` is the **query string**. Its named inputs are **query parameters**. Multiple parameters use `&`, as in `/add?a=2&b=3`.

The building comparison explains how an address is divided. In the application, these parts are parsed by software rather than physical desks.

## A5. Ordering without entering the kitchen

At a restaurant, the menu shows what can be ordered. The waiter accepts an order, communicates it to the kitchen, and brings back the result. The customer can ask for a meal without knowing the recipe, equipment, or arrangement of the kitchen.

The customer still needs to follow an understood interaction: choose an available item and provide the necessary choices. The kitchen can change how it prepares the meal while the ordering process remains familiar.

Software benefits from the same arrangement. One program needs a way to ask another program to do something without depending on every detail of how that work is implemented.

An **API**, Application Programming Interface, defines how software can interact. A **web API** makes operations available through HTTP requests and responses.

| Restaurant example | Software connection |
|---|---|
| Menu and ordering rules | Available operations and accepted inputs |
| Order passed through the waiter | Request made through the interface |
| Work in the kitchen | Internal implementation |
| Meal or explanation returned | Response containing a result or error |

The waiter is a useful image for the interaction. Technically, the API is the defined interface; it need not be a separate messenger program.

### From the analogy to an operation

For our API, an operation could be `GET /hello`. The caller needs to know its method, path, inputs, and possible responses. Together, these expectations form an **API contract**.

An **endpoint** here means a supported method and path. A **route** is the application's rule connecting a matching request to its handler, the code that processes it.

| Endpoint | Behaviour |
|---|---|
| `GET /hello` | Return a greeting; accept an optional name in the query string |
| `POST /greet` | Accept a name in a JSON body and return a greeting |

The caller uses these operations without needing to know the names of the Python functions behind them.

## A6. The destination does not describe the whole action

At the same service desk, someone might ask to read a saved note, replace its contents, change one line, or delete it. The destination can stay the same while the intended action changes.

An HTTP request expresses that intention through its **method**. The method and path are read together.

| Method | Usual purpose | Standalone example |
|---|---|---|
| `GET` | Retrieve a representation of data | Read note 7 |
| `POST` | Submit data for processing, often to create something | Submit text for a new note |
| `PUT` | Replace a resource's representation | Supply the complete replacement for note 7 |
| `PATCH` | Apply a partial update | Change only the title of note 7 |
| `DELETE` | Remove a resource | Delete note 7 |

A **resource** is something an API exposes, such as a note. These method meanings guide API design; the backend code must implement the behaviour.

Our lab uses only GET and POST. `POST /greet` processes a submitted name and returns a greeting. It stores nothing: using POST does not automatically create a database record.

Entering a URL in the browser's address bar sends GET. To submit a JSON body using POST, we will use the interactive `/docs` page or a PowerShell command. Supporting GET at a path does not automatically make POST available at that path.

## A7. Data needs labels and a shared format

Imagine receiving a slip containing only:

```text
Alex 2 yes
```

The values are readable, but their meaning is unclear. Labels make them usable: name is Alex, count is 2, enabled is yes. Software also needs precise rules for separating fields and recognising numbers, text, and lists.

**JSON**, JavaScript Object Notation, is a text format that supplies those rules. Many programming languages can read and write it.

```json
{
  "name": "Alex",
  "count": 2,
  "enabled": true,
  "labels": ["a", "b"],
  "note": null
}
```

An **object** uses `{}` and contains named fields. An **array** uses `[]` and contains a sequence of values. JSON supports strings, numbers, booleans, arrays, objects, and `null`. Keys and strings use double quotes; comments and trailing commas are not allowed.

### The body carries the submitted data

A request already identifies an action and destination. Its **body** can carry the information needed for that action, much like a completed form attached to a request.

For `POST /greet`, the request body is:

```json
{"name": "Alex"}
```

The response body contains:

```json
{"message": "Hello, Alex!"}
```

The header `Content-Type: application/json` describes the format of that body. JSON is one possible body format; HTTP also carries other kinds of content.

Python can hold the greeting in a dictionary. Before sending it, FastAPI converts it into JSON. The dictionary is an object in Python memory; JSON is text exchanged between programs. Python's `True`, `False`, and `None` correspond to JSON's `true`, `false`, and `null`.

## A8. Information belongs in an agreed place

Consider a form with a box labelled “name.” Writing a name on the outside of its envelope might be understandable to a person, but an automated system reading that box would still find it empty.

A web API similarly specifies where each input belongs. This allows the receiving software to find it consistently.

| Location | Example | What it supplies |
|---|---|---|
| Path parameter | `/numbers/7` matching `/numbers/{value}` | A value within the route's path |
| Query parameter | `/hello?name=Alex` | A named input after `?` |
| Request body | `{"name":"Alex"}` sent to POST `/greet` | Structured submitted data |
| Header | `Content-Type: application/json` | Information about the message's format |

For our GET route, `name` belongs in the query string. For our POST route, `name` belongs in the JSON body. Supplying it in the query string does not satisfy a required body field.

### Reading a form also involves checking it

A form may be missing a required field or contain text where a number is expected. Software needs to check such inputs before using them.

**Validation** checks data against declared requirements. Our POST endpoint declares that `name` is a required string. Sending `{}` fails because the field is missing. FastAPI reports the problem before calling the greeting function.

The requirements must be explicit. Declaring a string alone does not require it to be nonempty. Validation also cannot establish whether a supplied name is someone's real name; it checks the rules we define.

## A9. A reply can explain failure

A service desk can return several kinds of answer: “completed,” “that item does not exist,” or “the form is missing a required field.” All are replies. They report different outcomes.

Now consider arriving when nobody is listening at the desk. There is no reply at all. That is a different problem from receiving an explanation that the request could not be completed.

HTTP makes the same distinction. An HTTP response contains a **status code**, a three-digit summary of the outcome. Its body can provide details.

| Code | Meaning | Example |
|---|---|---|
| `200 OK` | Request handled successfully | A greeting was returned |
| `201 Created` | A resource was created | A new record was created |
| `400 Bad Request` | Server rejects the request as invalid | An application rejects inconsistent inputs |
| `404 Not Found` | Resource or route was not found | Requesting `/missing` |
| `405 Method Not Allowed` | Path does not accept this method | GET sent to our POST-only `/greet` |
| `422 Unprocessable Entity` | FastAPI could not validate the input | Missing `name` in the body |
| `500 Internal Server Error` | Unexpected failure inside the server | An unhandled exception in application code |

Broadly, `2xx` means success, `3xx` concerns redirection, `4xx` concerns the request, and `5xx` concerns the server. Status labels may be worded differently by different tools.

For example, a `404` from our API shows that the request reached a server that returned a response. “Connection refused” means no HTTP response was received from that address and port. The first check then is whether the server is running and the address is correct.

## A10. Making a Python function reachable

A Python function can already build a greeting:

```python
def hello():
    return {"message": "Hello!"}
```

Calling `hello()` from Python returns a dictionary. A browser cannot simply call that function by its Python name. Something must receive the browser's request, select this function, and send the returned value back in a form the browser can read.

We use two tools for these responsibilities. **Uvicorn** runs the server that listens for requests. **FastAPI** defines the application: which requests are supported, how inputs are checked, which function runs, and how its result becomes a response.

FastAPI is a **framework**, software that provides structure and reusable behaviour for building applications. Uvicorn runs it through a Python server interface called ASGI; its internal details are outside this lab.

Here is the greeting function connected to an HTTP route:

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/hello")
def hello():
    return {"message": "Hello!"}
```

1. `from fastapi import FastAPI` makes the framework's application class available.
2. `app = FastAPI()` creates the application object.
3. `@app.get("/hello")` registers the function below it for GET requests to `/hello`. This `@...` syntax is a **decorator**.
4. When a matching request arrives, FastAPI calls `hello()`.
5. The function returns a Python dictionary. FastAPI converts it to JSON and prepares a response, with status `200` by default.
6. Uvicorn sends the HTTP response back to the client.

The registered path is the address exposed to callers. The function name is internal to Python. Part B will start Uvicorn and make this sequence observable.

| Component | Role in the sequence |
|---|---|
| Browser or terminal | Sends the request and receives the response |
| Uvicorn | Listens for requests and runs the server |
| FastAPI | Matches the route, handles validation, and prepares the response |
| Python interpreter | Executes the Python code used by the application and tools |

## A11. Keeping tools, running state, and saved work organised

### Give each project its own Python packages

Our application uses code written by other developers. For example, FastAPI provides the tools we use to create API endpoints. We install this code as a **package**. A package our application needs is called a **dependency**.

Different projects may need different versions of the same package. If they share one installation, updating a package for one project could stop another project from working.

To avoid this, we create a **virtual environment** for each project. It lets us install the packages that project needs without changing the packages used by other projects.

We will use three things:

- **`.venv`** is the folder containing our project's virtual environment and its installed packages.
- **`pip`** is the tool we use to install packages.
- **`requirements.txt`** is the file where we list the packages and versions the project needs.

For example, this line in `requirements.txt` tells pip which version of FastAPI to install:

    fastapi==0.115.12

When someone else runs our project, they create their own virtual environment and install the packages listed in `requirements.txt`. They do not need to copy our `.venv` folder.

The virtual environment keeps Python packages separate. Our application code and database files stay in the project folders where we create them.

### Some information disappears when the program stops

When a Python program runs, it uses the computer's memory to hold values such as variables, lists, and dictionaries. Those values are temporary. If they exist only in memory, they are lost when the program stops.

For example, suppose our code starts with an empty list:

    names = []

While the program runs, we add `"Alex"` to that list. The list in memory now contains `"Alex"`, but the code in our `.py` file still says `names = []`. Changing the list while the program runs does not rewrite the source file.

If we stop and restart the program, Python runs the source code again and creates an empty list. To keep `"Alex"` for the next run, we must save it somewhere, such as a file or a database.

| Information | What happens when the server stops? |
|---|---|
| Code saved in a `.py` file | The file remains on disk. |
| A value added to a list only while the program runs | The value is lost. Restarting runs the code again. |
| Data successfully saved to a database file | The data remains and can be read after restarting. |

A **database** stores and organises data so an application can retrieve and update it later.

Our greeting API does not need a database yet. Each request supplies a name, and the server returns a greeting. It does not need to save that name for future requests. We will introduce a database when our application needs to keep information after the server stops.

### Keeping a history of the instructions

Editing a file and saving it replaces its current contents. Sometimes it is useful to inspect an earlier working version or see exactly what changed.

**Git** records versions of selected files in a local **repository**. A **commit** is a recorded snapshot with a message describing the change. **GitHub** hosts repositories online, making pushed commits available from another computer or to collaborators with access.

The source and setup instructions go into Git. Installed packages in `.venv` can be recreated. Uploading the source to GitHub stores the code; a server process must still run somewhere for the API to answer requests.

# Demo · The completed application

Open the completed demo and follow one existing interaction from input to visible result. Identify the interface, the request, the backend response, and any data that remains after restarting the app.

Return to this notebook for a separate practice API. Its examples use only greetings.

# Part B · Setup and first API
## B0. Tools and where instructions go

Use a Windows computer with internet access and permission to install software. Initial installations can be completed before the lesson.

| Tool | Purpose |
|---|---|
| Browser | GitHub, API URLs, and `/docs` |
| PowerShell | Runs the commands in this guide |
| Python 3.11 | Executes the example app |
| Git for Windows | Clones repositories and records changes |
| VS Code | Edits files and provides a terminal |
| GitHub account | Hosts the practice repository |

The examples use Python **3.11** and the package versions listed in B7. A current 3.11 patch release is suitable; the final patch number need not match an example. These are lab pins, not a claim about the newest package releases.

In Windows Start, search for **PowerShell** and open it. A line beginning with something like `PS C:\Users\Alex>` is the prompt. Type after it and press Enter. Do not type the prompt itself.

## B1. Install or verify Python

**PowerShell**

```powershell
py -3.11.3 --version
```

**Expected:** `Python 3.11.3 .x`, with a numeric patch version in place of `x`. If this works, continue to B2.

If Python 3.11.3  is missing:

1. Open [Python downloads](https://www.python.org/downloads/).
2. Install the **Python Install Manager** for Windows using the official installer or Microsoft Store option.
3. Close and reopen PowerShell so it can find the new commands.
4. Run:

```powershell
pymanager install 3.11.3
py -3.11.3  --version
```

`pymanager install 3.11.3 ` downloads the requested Python runtime. `py -3.11.3 ` selects that installed version. The older Windows `py` launcher also supports `py -3.11.3 `, but does not provide the install-manager subcommands.

If `py` is unavailable but an existing `python --version` reports 3.11.3 , use `python -m venv .venv` in B6. Otherwise use the install-manager route above. If both commands remain unavailable, see Troubleshooting.

Reference: [Python on Windows](https://docs.python.org/3/using/windows.html).

## B2. Install or verify Git and VS Code

**Git**

1. Run `git --version` in PowerShell. If a version appears, Git is available.
2. Otherwise download [Git for Windows](https://git-scm.com/downloads/win) and run its installer. Keep command-line access enabled and retain Git Credential Manager when offered.
3. Reopen PowerShell and run `git --version` again.

**VS Code**

1. Install [VS Code for Windows](https://code.visualstudio.com/download), using the Windows installer and enabling **Add to PATH** when offered.
2. Reopen PowerShell and run `code --version`. A version and other build details should appear.
3. In VS Code, open **Extensions** with `Ctrl+Shift+X`. Install **Python**, published by Microsoft. For opening this notebook in VS Code, also install **Jupyter**, published by Microsoft.

PATH is the list of folders Windows searches when a command names a program without its full location. Installers can update it; an already-open terminal may still have the older list.

The Python extension helps the editor understand Python. The interpreter installed in B1 is still required. GitHub access is checked in B4.

## B3. Find a working folder

**PowerShell**

```powershell
cd $HOME
pwd
ls
```

| Command | Meaning |
|---|---|
| `cd` | Change the current directory |
| `$HOME` | The current Windows user's home directory |
| `pwd` | Show the current directory |
| `ls` | List its contents; a PowerShell alias for `Get-ChildItem` |
| `mkdir` | Create a directory |
| `cd ..` | Move to the parent directory |
| `.` | The current directory |

If a folder named `KritonLabs` does not appear, create it:

```powershell
mkdir KritonLabs
```

Then enter it:

```powershell
cd KritonLabs
pwd
```

**Expected:** a path ending in `KritonLabs`. If it already exists, skip `mkdir` and enter it directly. Quote paths containing spaces, for example `cd "C:\Users\Alex\My Labs"`.

Relative paths are interpreted from the terminal's current directory. This is why the location of each command matters.

## B4. Create an empty GitHub repository and clone it

**Browser**

1. Sign in to [GitHub](https://github.com/). If needed, create an account and verify its email address.
2. Use **+ → New repository**. Select your own account as the owner.
3. Name it `fastapi-first-steps` and choose **Private**.
4. Leave README, `.gitignore`, and licence initialization disabled. Create the repository.
5. On the empty repository page, select **HTTPS** and copy the repository URL.

**PowerShell, inside `KritonLabs`**

Replace `YOUR-USERNAME` with your GitHub username, or paste the copied URL:

```powershell
git clone https://github.com/YOUR-USERNAME/fastapi-first-steps.git
cd fastapi-first-steps
git status
git remote -v
```

If Git opens a browser to authenticate, sign in to the account that owns the repository and complete the Git Credential Manager flow. An ordinary GitHub account password is not accepted as an HTTPS Git password. If authentication stops at a terminal password prompt, use the Git troubleshooting instructions below.

**Expected:** Git may warn that the cloned repository is empty. That is expected. `git status` should say there are no commits yet, and `git remote -v` should show your GitHub URL under `origin`.

A **repository** is a project tracked by Git. A **remote** is a named connection to another repository. `clone` creates the local project folder and configures its remote. Do not run `git init` or create another `fastapi-first-steps` folder inside it. `git init` is used for the alternative workflow of starting Git in an existing local folder.

## B5. Open the project and create `.gitignore`

**PowerShell, inside `fastapi-first-steps`**

```powershell
code .
```

If that command is unavailable, open VS Code and use **File → Open Folder** to select `fastapi-first-steps`. If Workspace Trust appears, trust the folder you just created.

Open **Terminal → New Terminal**. Select **PowerShell** from the terminal profile menu if another shell opens. Run `pwd` and confirm the path ends in `fastapi-first-steps`. This is the **repository root**, the top-level project folder. Run the remaining project commands here.

In VS Code's Explorer, click **New File**, enter `.gitignore`, and paste the following content. Save with `Ctrl+S`.

**File: `.gitignore`**

```gitignore
# Installed Python packages belong to each developer's environment.
.venv/

# Python creates these files when it imports and runs code.
__pycache__/
*.pyc

# Automated tests create this cache.
.pytest_cache/

# Local database files contain changing application data.
data/*.db*
```

This tells Git which untracked files to leave out of commits. It does not delete files. `*.pyc` matches generated bytecode files; `.py` source files remain trackable. The test and database patterns are for later work; this lab creates neither.

**Check:** the Explorer shows `.gitignore` at the repository root, with exactly that name. Create files through VS Code to avoid accidentally adding a `.txt` suffix.

## B6. Create and inspect the virtual environment

**PowerShell, repository root**

```powershell
py -3.11 -m venv .venv
.\.venv\Scripts\python.exe --version
.\.venv\Scripts\python.exe -c "import sys; print(sys.executable)"
```

If B1 established that only `python` is available and it is version 3.11, replace the first command with `python -m venv .venv`.

| Part | Meaning |
|---|---|
| `py -3.11` | Select Python 3.11 |
| `-m venv` | Run Python's virtual-environment module |
| `.venv` | Name of the environment folder to create |
| `.\` | Begin at the current directory |
| `.venv\Scripts\python.exe` | Path to this environment's Python executable |
| `--version` | Print the Python version |
| `-c` | Execute the short Python code supplied after it |

A backslash separates Windows path components. `.venv` is the folder name; its initial dot alone does not mean “current directory.”

**Expected:** Python reports `3.11.x`; the executable path ends in `fastapi-first-steps\.venv\Scripts\python.exe`. Creating the environment may otherwise produce no output.

Use this explicit executable path for package installation and running the app. Activation is optional and is not needed for these commands. No PowerShell execution-policy change is required.

In VS Code, press `Ctrl+Shift+P`, select **Python: Select Interpreter**, and choose this `.venv` interpreter. If absent, use **Enter interpreter path** to locate `.venv\Scripts\python.exe`. This selection configures editor features; the commands above already select the interpreter explicitly.

## B7. Declare and install dependencies

Create and save `requirements.txt` at the repository root.

**File: `requirements.txt`**

```text
# FastAPI defines endpoints, validates inputs, and generates API documentation.
fastapi==0.115.12

# Uvicorn runs the server that receives HTTP requests.
uvicorn==0.34.2
```

`==` fixes the version of the named package. Lines starting with `#` are comments. These packages also install their own dependencies, including Pydantic, which FastAPI uses for validation. This short file pins the two direct dependencies; it is not a complete lock of every indirect dependency.

**PowerShell, repository root**

```powershell
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
.\.venv\Scripts\python.exe -m pip check
.\.venv\Scripts\python.exe -c "import fastapi, uvicorn; print(fastapi.__version__, uvicorn.__version__)"
```

`-m pip` runs pip using the selected Python. `install` installs packages; `-r` reads them from the named file.

**Expected:** installation completes without an error, `pip check` reports no broken requirements, and the final command prints `0.115.12 0.34.2`. Additional dependency names in installation output are normal. A notice that a newer pip is available is not an installation failure.

## B8. Create the first Python file

**PowerShell, repository root**

```powershell
mkdir app
```

In VS Code's Explorer, create two files inside `app`:

| File | Content or purpose |
|---|---|
| `app/__init__.py` | Leave empty. Marks `app` as a regular Python package. There are two underscores on each side of `init`. |
| `app/main.py` | Contains the application below. |

**File: `app/main.py`**

```python
from fastapi import FastAPI

app = FastAPI(title="First API")


@app.get("/hello")
def hello(name: str = "World"):
    return {"message": f"Hello, {name}!"}
```

Save the file. Python indentation matters; use four spaces inside the function, as shown.

`name: str` declares a string parameter. `= "World"` supplies its default. For this route, FastAPI reads `name` from the query string because it is a simple parameter absent from the path. An `f"..."` string inserts the value inside `{name}` into the message.

**Check:** `main.py` is inside `app`; `.gitignore` and `requirements.txt` are at the root. There is only one `app` folder.

## B9. Start the server

**PowerShell, repository root**

```powershell
.\.venv\Scripts\python.exe -m uvicorn app.main:app --reload --port 8766
```

| Part | Meaning |
|---|---|
| `-m uvicorn` | Run Uvicorn with this environment's Python |
| `app.main` | Import the module `app/main.py` |
| `:app` | Use the object named `app` defined in that module |
| `--reload` | Restart the application after Python source changes during development |
| `--port 8766` | Listen on port 8766 |

Uvicorn binds to the loopback address by default. The server is available on this computer.

**Expected:** output includes a URL such as `http://127.0.0.1:8766` and indicates application startup completed. Exact log lines and process numbers vary.

Leave this terminal running. It shows server logs rather than a new PowerShell prompt. Use a second terminal for additional commands. If there is an error, fix it using Troubleshooting before continuing.

Do not start this app with the editor's **Run Python File** button. That runs the file without starting Uvicorn to serve requests.

## B10. Send GET requests in the browser

**Browser address bar**

Open:

```text
http://127.0.0.1:8766/hello
```

**Expected JSON**

```json
{"message": "Hello, World!"}
```

Then open:

```text
http://127.0.0.1:8766/hello?name=Alex
```

**Expected JSON**

```json
{"message": "Hello, Alex!"}
```

The browser may format the JSON differently. The values should match. The server terminal records these requests with a `200` status.

Opening `http://127.0.0.1:8766/` returns `404` because no `/` route has been defined. It does not mean that `/hello` or the server is broken.

## B11. Add a POST route with a request body

Replace `app/main.py` with this complete version and save it.

**File: `app/main.py`**

```python
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="First API")


class GreetingInput(BaseModel):
    name: str


@app.get("/hello")
def hello(name: str = "World"):
    return {"message": f"Hello, {name}!"}


@app.post("/greet")
def greet(body: GreetingInput):
    return {"message": f"Hello, {body.name}!"}
```

`GreetingInput` describes the expected body. `BaseModel` provides Pydantic's validation behaviour. `name: str` declares a required string field because no default is supplied.

`body: GreetingInput` tells FastAPI to read a JSON body and validate it against this model. After validation, `body.name` accesses its value. The Python class syntax is used here to declare the fields; no additional object-oriented design is needed for this lab.

The POST handler returns status `200`: it computes a response and creates no stored resource. The server should reload after saving. If a syntax error appears, fix the file and save again.

**Check:** the server returns to a running state. Keep it running for the next steps.

## B12. Use the interactive API documentation

**Browser**

Open `http://127.0.0.1:8766/docs`.

This is a web client generated for the API. It documents available operations and sends real requests when **Execute** is pressed. It is separate from a custom application frontend.

1. Expand **GET /hello**.
2. Click **Try it out**. Enter `Alex` for `name` and click **Execute**.
3. Under **Server response**, read code `200` and the greeting body.
4. Expand **POST /greet** and click **Try it out**.
5. Replace the example body with `{"name": "Alex"}` and click **Execute**.
6. Confirm code `200` and `{"message":"Hello, Alex!"}`.

The **Request URL** shows where the request went. The response body shows returned data. The documented **Responses** section describes possible responses; the **Server response** section shows the actual result of this execution.

FastAPI's docs page normally loads interface assets from a CDN, so internet access may be needed to display it even while the API runs locally.

## B13. Read known error responses

Follow these checks to see the difference between a missing route, a wrong method, and invalid input.

| Action | Expected result | Explanation |
|---|---|---|
| Open `/missing` in the browser | `404`, `{"detail":"Not Found"}` | No route matches this path |
| Open `/greet` in the address bar | `405`, `{"detail":"Method Not Allowed"}` | Address-bar navigation sends GET; this path accepts POST |
| In `/docs`, execute POST `/greet` with `{}` | `422`, a validation error body | Required `name` field is absent |

For the `422` response, the error location includes `body` and `name`. Exact wording and additional fields may differ across validation-library versions. The handler does not run when body validation fails.

After the error check, submit `{"name":"Alex"}` again. The response should return to `200`. Error responses are feedback about individual requests; these examples do not stop the server.

## B14. Send requests from PowerShell

In VS Code, use the **+** button in the terminal panel to open a second PowerShell terminal. Keep the server terminal running.

We will use **`Invoke-RestMethod`**, a built-in PowerShell command that sends an HTTP request and reads the server's response. Here, PowerShell acts as the **client**: it sends requests to our running API.

These commands can run from any directory because they contact the server through its address.

### Send a GET request

```powershell
Invoke-RestMethod -Uri 'http://127.0.0.1:8766/hello?name=Alex' -Method Get
```

The words beginning with `-` are **command parameters**. They tell PowerShell how to send the request.

| Part | What it means |
|---|---|
| `Invoke-RestMethod` | The command that sends the request and reads the response. |
| `-Uri` | Specifies the destination URL. |
| `http://127.0.0.1:8766` | The address of our server, running on this computer at port `8766`. |
| `/hello` | The path of the operation we want to call. |
| `?name=Alex` | A query parameter that supplies the name `Alex` in the URL. |
| `-Method Get` | Selects `GET` as the HTTP request method. |

This request supplies the name in the URL, so we do not send a request body.

**Expected:** a result with `message` equal to `Hello, Alex!`.

The server returns JSON. `Invoke-RestMethod` converts that JSON into a PowerShell object whose fields we can read. PowerShell may display it as a small table:

```text
message
-------
Hello, Alex!
```

### Send a POST request

This time, we send the name in the request body:

```powershell
Invoke-RestMethod -Uri 'http://127.0.0.1:8766/greet' -Method Post -ContentType 'application/json' -Body '{"name":"Alex"}'
```

| Part | What it means |
|---|---|
| `-Uri 'http://127.0.0.1:8766/greet'` | Sends the request to the `/greet` path on our server. |
| `-Method Post` | Selects `POST` as the HTTP request method. |
| `-ContentType 'application/json'` | Adds the HTTP header `Content-Type: application/json`, telling the server that the body uses JSON. |
| `-Body '{"name":"Alex"}'` | Supplies the request body: JSON containing the name `Alex`. |

A **header** gives the receiver information about the message. Here, `Content-Type` describes the format of the data we are sending. The **body** contains the data itself.

`-ContentType` is a PowerShell command parameter. PowerShell uses its value to create the HTTP `Content-Type` header. We do not write the header separately.

Single quotes surround the body in the command so PowerShell keeps the JSON double quotes intact. The outer single quotes are not sent to the server.

**Expected:** `message` equals `Hello, Alex!` again.

We have now called the same API using different clients: the browser, the interactive `/docs` page, and PowerShell.

### Inspect the response status and body

`Invoke-RestMethod` is convenient when we want to read the returned data. To inspect the status code and the response body as text, use another built-in PowerShell command, **`Invoke-WebRequest`**:

```powershell
$reply = Invoke-WebRequest -UseBasicParsing -Uri 'http://127.0.0.1:8766/hello' -Method Get
$reply.StatusCode
$reply.Content
```

The first line sends the request and stores the response in a variable named `$reply`. The next two lines read parts of that response:

| Expression | What it shows | Expected value |
|---|---|---|
| `$reply.StatusCode` | The HTTP status code returned by the server. | `200` |
| `$reply.Content` | The response body as text. | `{"message":"Hello, World!"}` |

This request does not include a `name` query parameter, so the API uses its default name, `World`.

The response also has headers. To inspect them, run:

```powershell
$reply.Headers
```

Look for `Content-Type`. For this response, its value should be `application/json`, telling us that the returned body uses JSON.

`-UseBasicParsing` tells Windows PowerShell to read the response without using its older browser-based HTML parser. Include it as shown.

Use the command names `Invoke-RestMethod` and `Invoke-WebRequest` as written. The shorter name `curl` can refer to different commands on different Windows installations.

## B15. Edit, stop, and restart

**Edit and reload**

In the GET function, change only `World` to `Everyone`, then save `app/main.py`. Refresh `/hello` without a query string. The response becomes `{"message":"Hello, Everyone!"}`. A request with `?name=Alex` still uses `Alex`.

Restore `World` and save so the file matches B11.

**Stop**

Select the terminal running Uvicorn and press `Ctrl+C`. Wait for the PowerShell prompt to return. A new browser request to the API can no longer connect.

**Restart**

From the repository root, run:

```powershell
.\.venv\Scripts\python.exe -m uvicorn app.main:app --reload --port 8766
```

Open `/hello` again to confirm the API is reachable. Source files and installed packages remain on disk after stopping; the server process must be started again when needed. `--reload` is a development convenience, not a deployment configuration.

## B16. Document how to run the app

Create `README.md` at the repository root. Paste the content below and save it. Markdown uses `#` for headings and backticks for code formatting.

**File: `README.md`**

````markdown
# First API

Practice API using Python 3.11, FastAPI, and Uvicorn.

## Setup on Windows PowerShell

Open the cloned repository folder, then run:

```powershell
py -3.11 -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
```

## Run

```powershell
.\.venv\Scripts\python.exe -m uvicorn app.main:app --reload --port 8766
```

Open http://127.0.0.1:8766/docs in the browser.
Stop the server with Ctrl+C in its terminal.

## Endpoints

- GET /hello: optional `name` query parameter, default `World`.
- POST /greet: JSON body containing a required string field `name`.

Both return a JSON greeting. No data is stored.
````

**Check:** the README contains the setup command, server command, documentation URL, and endpoint descriptions. `.venv` is recreated locally; it is not part of the source to upload.

## B17. Commit and push the source code

Use the second PowerShell terminal, at the repository root. First configure the identity attached to commits in this repository. Replace the example values with your own.

```powershell
git config user.name "Your Name"
git config user.email "YOUR-GITHUB-COMMIT-EMAIL"
```

Use an email connected to your GitHub account, or copy the exact private `noreply` address shown in **GitHub → Settings → Emails**. These commands label commits; they do not sign in to GitHub. Without `--global`, they apply to this repository only.

Then run:

```powershell
git status
git add .gitignore requirements.txt README.md app/__init__.py app/main.py
git diff --cached --stat
git commit -m "Add first FastAPI example"
git branch -M main
git push -u origin main
```

| Command | Meaning |
|---|---|
| `status` | Show tracked and untracked changes |
| `add` | Stage selected files for the next commit |
| `diff --cached --stat` | Summarise the staged changes |
| `commit` | Record a local snapshot with a message |
| `branch -M main` | Name the current branch `main` |
| `push -u origin main` | Upload it and connect the local branch to the remote branch |

A **branch** is a named line of development. This lab uses only `main`.

**Check before committing:** the staged files are the five listed files. `.venv`, `__pycache__`, and `.pyc` files should be absent.

**Check after pushing:** refresh the GitHub repository page. The five source/documentation files should appear, with the two Python files inside `app`. Run `git status`; it should report a clean working tree and an up-to-date branch.

The repository now contains reproducible source and instructions. The backend still runs on your own computer while Uvicorn is active.

## B18. Resume work next time

1. Open the `fastapi-first-steps` folder in VS Code.
2. Open a PowerShell terminal. Use `pwd` to verify the repository root.
3. If continuing from changes already pushed elsewhere, run `git pull` while your working tree is clean. If Git reports a conflict, resolve it before continuing; do not discard local work to bypass the message.
4. Start the server:

```powershell
.\.venv\Scripts\python.exe -m uvicorn app.main:app --reload --port 8766
```

5. Open `http://127.0.0.1:8766/docs`.

The environment and packages need not be recreated each session. On a new computer, clone the repository, create `.venv`, and install `requirements.txt` as described in B6–B7.

After editing and saving tracked files:

```powershell
git status
git add app/main.py
git commit -m "Describe the change"
git push
```

Stage the actual files changed; the example above stages only `app/main.py`. Saving a file, committing it, and pushing it are three separate actions.

# Troubleshooting
## Software and folders

| Symptom | Check and action |
|---|---|
| `py` or `pymanager` is not recognised | Close and reopen PowerShell after installation. Confirm Python Install Manager is installed. Use the official Windows troubleshooting link below if aliases still fail. |
| `python` opens the Microsoft Store | Use the verified `py -3.11` command. If it is also missing, complete B1. |
| `git` is not recognised | Install Git for Windows with command-line access, then reopen VS Code and its terminal. |
| `code` is not recognised | Use VS Code's File → Open Folder. Reopen the terminal after installation to refresh PATH. |
| `.venv\Scripts\python.exe` cannot be found | Run `pwd` and `ls`. Return to the repository root. If `.venv` was never created, run B6. |
| `requirements.txt` cannot be found | Verify the current directory and save the file at the root. Check that it is not named `requirements.txt.txt`. |
| `Activate.ps1` is blocked | Activation is unnecessary here. Use the explicit `.\.venv\Scripts\python.exe` commands. |
| VS Code underlines an installed import | Select the `.venv` interpreter in B6. Check the import using B7's terminal command. |
| Package download fails | Read the final error. Check internet access and any managed-network restrictions, then rerun installation. A Python-version or build error should be addressed using the course's Python 3.11 environment. |

Python installation details: [official Windows troubleshooting](https://docs.python.org/3/using/windows.html#troubleshooting).

## Server and requests

| Symptom | Check and action |
|---|---|
| `No module named uvicorn` or `fastapi` | Run B7 using the `.venv` executable. Installing globally does not install into this environment. |
| Cannot import `app.main` | Run from the repository root. Verify `app/main.py`, `app/__init__.py`, and the spelling `app.main:app`. |
| Attribute `app` not found | Check that `main.py` defines `app = FastAPI(...)` at the top level and is saved. |
| `SyntaxError` or `IndentationError` | Read the filename and line number in the terminal. Compare with B11, fix quotes/indentation, and save. |
| Port is already in use, or Windows error 10048 | Stop an earlier copy of your server with Ctrl+C. If the port is used by another application, start with `--port 8767` and change every client URL to 8767. |
| Browser cannot connect | Check that Uvicorn is running, startup completed, and the URL uses `http://127.0.0.1` with the chosen port. |
| `/docs` is blank but `/hello` works | Check internet access to the documentation page's external assets. The API can still be tested through PowerShell. |
| `/docs` does not list the POST route | Save `main.py`, check reload logs, and refresh `/docs`. Confirm you started the server from the intended repository. |
| `404` | Check the path. `/` is deliberately undefined. |
| `405` for `/greet` | Send POST through `/docs` or PowerShell; the address bar sends GET. |
| `422` | Read the error location. Submit the exact JSON `{"name":"Alex"}` with the JSON content type. |
| `500` | Read the traceback in the server terminal. It identifies the failing Python code. |

## Git and GitHub

| Symptom | Check and action |
|---|---|
| Empty-repository warning after clone | Expected for this lab. Continue inside the newly created folder. |
| Clone reports that the destination already exists | Inspect that folder. If it is your existing clone, open it rather than cloning again. |
| `Repository not found` | Copy the HTTPS URL from your repository page and verify the signed-in account has access. Private repositories require authentication. |
| HTTPS authentication fails | Use the Git Credential Manager browser flow included with Git for Windows. If it is missing, rerun the Git installer and enable it. Use GitHub's HTTPS authentication guide below if a terminal credential flow is required; use a token as documented, not an account password. Never put a token in a source file or remote URL. |
| `Author identity unknown` | Complete the two `git config` commands in B17, then retry the commit. |
| `src refspec main does not match any` | Ensure the first commit succeeded, then run `git branch -M main` and push again. |
| Push rejected because the remote contains changes | This can happen if the GitHub repo was initialized with files. Inspect local and remote histories; do not force-push. Keep local work and get help reconciling the histories. |
| `.venv` appears staged | Before the first commit, run `git rm -r --cached -- .venv`, check `.gitignore`, and stage the five intended files again. This removes the directory from Git's index while leaving it on disk. |
| GitHub does not show saved edits | Saving is local. Stage, commit, and push the changes, then refresh the page. |

Reference: [GitHub HTTPS authentication](https://docs.github.com/en/get-started/git-basics/caching-your-github-credentials-in-git).

# Completion check

- The repository contains `.gitignore`, `requirements.txt`, `README.md`, `app/__init__.py`, and `app/main.py`.
- The environment's Python executable is inside `.venv`.
- Uvicorn starts from the repository root.
- GET `/hello` returns the default greeting; `?name=Alex` changes it.
- POST `/greet` accepts a JSON body in `/docs` and PowerShell.
- A missing `name` produces `422`; an unknown path produces `404`.
- The server can be stopped and restarted.
- GitHub contains the source files and instructions, without the local environment.

# References

- [Python on Windows and Python Install Manager](https://docs.python.org/3/using/windows.html)
- [Python virtual environments](https://docs.python.org/3/library/venv.html)
- [Git for Windows](https://git-scm.com/downloads/win)
- [VS Code on Windows](https://code.visualstudio.com/docs/setup/windows)
- [Python environments in VS Code](https://code.visualstudio.com/docs/python/environments)
- [Creating a GitHub repository](https://docs.github.com/en/repositories/creating-and-managing-repositories/creating-a-new-repository)
- [Cloning a repository](https://docs.github.com/en/repositories/creating-and-managing-repositories/cloning-a-repository)
- [FastAPI first steps](https://fastapi.tiangolo.com/tutorial/first-steps/)
- [FastAPI request bodies](https://fastapi.tiangolo.com/tutorial/body/)
- [Uvicorn settings](https://www.uvicorn.org/settings/)

Examples prepared for Python 3.11, FastAPI 0.115.12, and Uvicorn 0.34.2. The setup path uses Windows PowerShell. The example API is independent of the later project.